# Week 30, Day 5: Composition versus inheritance

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w030d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Queue(list):
    def join(self, name):
        self.append(name)

    def serve(self):
        return self.pop(0)

q = Queue()
q.join("Ada")
q.join("Bo")
q.insert(0, "Cheeky")
print(q.serve(), type(q + ["Dan"]).__name__, len(q))

In [ ]:
class Queue:
    def __init__(self):
        self._people = []                 # the queue HAS a list

    def join(self, name):
        self._people.append(name)

    def serve(self):
        if not self._people:
            raise IndexError("the queue is empty")
        return self._people.pop(0)

    def size(self):
        return len(self._people)

q = Queue()
q.join("Ada")
q.join("Bo")
print(q.serve(), q.size())
print(hasattr(q, "insert"), isinstance(q, list))

In [ ]:
class Carriage:
    def __init__(self, number, seats):
        self.number = number
        self.seats = seats
        self.booked = 0

    def free(self):
        return self.seats - self.booked

class Train:
    def __init__(self, code, carriages):
        self.code = code
        self.carriages = list(carriages)

    def free_seats(self):
        return sum(c.free() for c in self.carriages)

    def book(self):
        for c in self.carriages:
            if c.free() > 0:
                c.booked += 1
                return c.number
        raise ValueError("train full")

t = Train("1S12", [Carriage("A", 2), Carriage("B", 40)])
print([t.book() for _ in range(3)], t.free_seats())

In [ ]:
class FullPrice:
    def price(self, total):
        return total

class StudentPrice:
    def price(self, total):
        return total - total // 10

class BulkPrice:
    def __init__(self, over, percent):
        self.over = over
        self.percent = percent

    def price(self, total):
        if total > self.over:
            return total - total * self.percent // 100
        return total

class Order:
    def __init__(self, items, pricing=None):
        self.items = list(items)
        self.pricing = pricing if pricing is not None else FullPrice()

    def total(self):
        return self.pricing.price(sum(pence for _, pence in self.items))

order = Order([("Textbook", 2400), ("Pens", 300)])
print(order.total())
order.pricing = StudentPrice()             # the customer shows a student card
print(order.total())
print(Order(order.items, BulkPrice(2000, 15)).total())

In [ ]:
class LowerDict(dict):
    """A dictionary whose keys are always lower case. Or is it?"""

    def __setitem__(self, key, value):
        super().__setitem__(key.lower(), value)

    def __getitem__(self, key):
        return super().__getitem__(key.lower())

d = LowerDict({"Lagos": 1})
d["LEEDS"] = 2
d.update({"Kano": 3})
print(d)
print(d.get("LEEDS"), "leeds" in d)

In [ ]:
class LowerDict:
    def __init__(self, data=None):
        self._data = {}
        for key, value in (data or {}).items():
            self.set(key, value)

    def set(self, key, value):
        self._data[key.lower()] = value

    def get(self, key, default=None):
        return self._data.get(key.lower(), default)

    def has(self, key):
        return key.lower() in self._data

d = LowerDict({"Lagos": 1})
d.set("LEEDS", 2)
print(d.get("lagos"), d.get("Leeds"), d.has("KANO"))

In [ ]:
class Screen:
    def send(self, text):
        print(text)

class Shouting:
    def __init__(self, inner):
        self.inner = inner

    def send(self, text):
        self.inner.send(text.upper())

class Signed:
    def __init__(self, inner, name):
        self.inner = inner
        self.name = name

    def send(self, text):
        self.inner.send(f"{text} ({self.name})")

Screen().send("Match postponed")
Shouting(Screen()).send("Match postponed")
Signed(Shouting(Screen()), "Coach").send("Match postponed")
Shouting(Signed(Screen(), "Coach")).send("Match postponed")

> **Key idea.** Composition builds an object from parts held as attributes and delegates work to them: a queue has a list, a car has an engine. It exposes only the operations you choose, lets parts be replaced while the program runs (a pricing strategy), and avoids a class for every combination. Inheriting for convenience gives the child unwanted methods, ties it to the parent's insides, and with built-in types like `dict` your overrides may be skipped. Wrappers hold an object with the same interface and add to it. Prefer composition; inherit when "every X is a Y, and works wherever a Y does" is true.

### Exercise: A stack that is only a stack

Write `Stack` by composition (it must **not** inherit from `list`). It has `push(item)`, `pop()` (removes and returns the top item, raising `IndexError("empty stack")` when empty), `peek()` (returns the top item without removing it, same error when empty) and `size()`. It must have no `insert`, `append` or `sort`.

In [ ]:
class Stack:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stack():\n    \"a stack\"\n    S = ns[\"Stack\"]\n    s = S()\n    s.push(\"plate 1\")\n    s.push(\"plate 2\")\n    assert s.peek() == \"plate 2\" and s.size() == 2\n    assert s.pop() == \"plate 2\" and s.pop() == \"plate 1\" and s.size() == 0\n    for f in (s.pop, s.peek):\n        try:\n            f()\n            assert False, \"an empty stack should raise IndexError\"\n        except IndexError as err:\n            assert str(err) == \"empty stack\", f\"message was {str(err)!r}\"\ndef test_design():\n    \"composition\"\n    S = ns[\"Stack\"]\n    assert not issubclass(S, list), \"build Stack with a list inside, not as a list\"\n    s = S()\n    for name in [\"insert\", \"append\", \"sort\"]:\n        assert not hasattr(s, name), f\"a stack shouldn't have {name}\"\n    a, b = S(), S()\n    a.push(1)\n    assert b.size() == 0, \"each stack needs its own list\"")

### Exercise: Swappable fare rules

A `Journey(miles, rule)` works out its fare by asking its `rule` part: `fare()` returns `rule.fare(miles)`. Write three rules:

- `Flat(pence)`: the same fare whatever the distance;
- `PerMile(pence_per_mile, minimum)`: `pence_per_mile * miles`, but never less than `minimum`;
- `Capped(inner, cap)`: wraps another rule and returns its fare, but never more than `cap`.

Then write `Journey`. Changing `journey.rule` must change the fare.

In [ ]:
___

class Journey:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rules():\n    \"the rules\"\n    F, P, C = ns[\"Flat\"], ns[\"PerMile\"], ns[\"Capped\"]\n    assert F(250).fare(1) == 250 and F(250).fare(90) == 250\n    assert P(30, 200).fare(10) == 300 and P(30, 200).fare(2) == 200\n    assert C(P(30, 200), 1000).fare(50) == 1000 and C(P(30, 200), 1000).fare(10) == 300\n    assert C(C(F(900), 800), 500).fare(1) == 500\ndef test_journey():\n    \"journeys\"\n    J, F, P = ns[\"Journey\"], ns[\"Flat\"], ns[\"PerMile\"]\n    j = J(12, P(25, 100))\n    assert j.fare() == 300\n    j.rule = F(150)\n    assert j.fare() == 150, \"changing the rule should change the fare\"\n    class Free:\n        def fare(self, miles):\n            return 0\n    assert J(500, Free()).fare() == 0, \"any object with fare(miles) should work as a rule\"")

### Exercise: Channels and wrappers

Messages go out through channels, each with a `send(text)` method. Write:

- `Outbox`: a channel that keeps every text it's sent in a list `sent`;
- `Sms(inner)`: a wrapper that passes on only the first 160 characters of the text;
- `Prefixed(inner, prefix)`: a wrapper that passes on `prefix + text`;
- `Notifier(channels)`, which keeps its own **copy** of the list of channels; `notify(text)` sends the text to every channel and returns how many it sent to. `add(channel)` adds another.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_wrappers():\n    \"wrappers\"\n    O, S, P = ns[\"Outbox\"], ns[\"Sms\"], ns[\"Prefixed\"]\n    box = O()\n    P(S(box), \"[Club] \").send(\"x\" * 200)\n    S(P(box, \"[Club] \")).send(\"y\" * 200)\n    assert box.sent[0] == (\"[Club] \" + \"x\" * 200)[:160], \"Prefixed outside Sms: the prefix is added, then the whole text shortened\"\n    assert box.sent[1] == \"[Club] \" + \"y\" * 160, \"Sms outside Prefixed: the text is shortened, then the prefix added\"\n    S(box).send(\"short\")\n    assert box.sent[2] == \"short\"\ndef test_notifier():\n    \"the notifier\"\n    N, O, P = ns[\"Notifier\"], ns[\"Outbox\"], ns[\"Prefixed\"]\n    a, b = O(), O()\n    chans = [a, P(b, \"!\")]\n    n = N(chans)\n    chans.append(O())\n    assert n.notify(\"Training at 6\") == 2, \"keep a copy of the list you're given\"\n    assert a.sent == [\"Training at 6\"] and b.sent == [\"!Training at 6\"]\n    c = O()\n    n.add(c)\n    assert n.notify(\"Cancelled\") == 3 and c.sent == [\"Cancelled\"] and N([]).notify(\"x\") == 0")

### Exercise: A league of teams

Write `Team(name)` with `played`, `points`, `scored` and `conceded`, all starting at 0, and `League(names)`, which **has** a `Team` for each name. `League.result(home, home_goals, away_goals, away)` updates both teams (3 points for a win, 1 each for a draw), raising `KeyError` for a team not in the league and `ValueError` if a team plays itself. `League.table()` returns the teams' names in order: most points first, then the bigger goal difference (scored minus conceded), then more goals scored, then name A to Z.

In [ ]:
class Team:
    ___

class League:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_table():\n    \"results and the table\"\n    L, T = ns[\"League\"], ns[\"Team\"]\n    lg = L([\"Rangers\", \"Enyimba\", \"Hearts\", \"Pillars\"])\n    lg.result(\"Rangers\", 2, 0, \"Enyimba\")\n    lg.result(\"Hearts\", 1, 1, \"Pillars\")\n    lg.result(\"Enyimba\", 3, 1, \"Hearts\")\n    lg.result(\"Pillars\", 0, 2, \"Rangers\")\n    assert lg.table() == [\"Rangers\", \"Enyimba\", \"Hearts\", \"Pillars\"], f\"got {lg.table()}\"\n    lg2 = L([\"B\", \"A\", \"C\"])\n    assert lg2.table() == [\"A\", \"B\", \"C\"]\ndef test_teams():\n    \"the teams inside\"\n    L, T = ns[\"League\"], ns[\"Team\"]\n    lg = L([\"Rangers\", \"Enyimba\"])\n    lg.result(\"Rangers\", 2, 2, \"Enyimba\")\n    team = T(\"Kano Pillars\")\n    assert (team.name, team.played, team.points, team.scored, team.conceded) == (\"Kano Pillars\", 0, 0, 0, 0)\n    assert not issubclass(L, (list, dict)), \"League should have teams, not be a list or dictionary\"\ndef test_errors():\n    \"errors\"\n    L = ns[\"League\"]\n    lg = L([\"Rangers\", \"Enyimba\"])\n    try:\n        lg.result(\"Rangers\", 1, 0, \"Arsenal\")\n        assert False, \"an unknown team should raise KeyError\"\n    except KeyError:\n        pass\n    try:\n        lg.result(\"Rangers\", 1, 0, \"Rangers\")\n        assert False, \"a team can't play itself\"\n    except ValueError:\n        pass\n    assert lg.table() == [\"Enyimba\", \"Rangers\"], \"failed results shouldn't change anything\"")

### Bug hunt: A class, not an object

The order crashes with `TypeError: StudentPrice.price() missing 1 required positional argument: 'total'`. It should print `2430`. Fix the one line that's wrong.

In [ ]:
class StudentPrice:
    def price(self, total):
        return total - total // 10

class Order:
    def __init__(self, items, pricing):
        self.items = list(items)
        self.pricing = pricing

    def total(self):
        return self.pricing.price(sum(self.items))

order = Order([2400, 300], StudentPrice)
print(order.total())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the total\"\n    expect_output(\"2430\")")

### Bug hunt: Shared players

Two teams are made, but both end up with all three players. It should print `['Ada', 'Bo'] ['Chidi']`. Fix `Squad`.

In [ ]:
class Squad:
    def __init__(self, name, players=[]):
        self.name = name
        self.players = players

    def sign(self, player):
        self.players.append(player)

reds = Squad("Reds")
blues = Squad("Blues")
reds.sign("Ada")
reds.sign("Bo")
blues.sign("Chidi")
print(reds.players, blues.players)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate squads\"\n    expect_output(\"['Ada', 'Bo'] ['Chidi']\")\n    given = [\"X\"]\n    s = ns[\"Squad\"](\"Greens\", given)\n    s.sign(\"Y\")\n    assert given == [\"X\"], \"a squad shouldn't change the list it was given\"")

### Bug hunt: Inherited too much

`Playlist` inherits from `list`, so `+` hands back a plain list and the combined playlist has lost its `name` and its `duration` method: this crashes with `AttributeError`. Rebuild `Playlist` by composition so the program prints `Party 9`. Keep `add`, `duration` and `+` working (`__add__` should return a new `Playlist` named after the left one, holding the songs of both).

In [ ]:
class Playlist(list):
    def __init__(self, name):
        super().__init__()
        self.name = name

    def add(self, title, minutes):
        self.append((title, minutes))

    def duration(self):
        return sum(m for _, m in self)

a = Playlist("Party")
a.add("Ye", 4)
b = Playlist("Chill")
b.add("Calm Down", 3)
b.add("Essence", 2)
both = a + b
print(both.name, both.duration())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the combined playlist\"\n    expect_output(\"Party 9\")\n    P = ns[\"Playlist\"]\n    assert not issubclass(P, list), \"use composition, not inheritance from list\"\n    x, y = P(\"x\"), P(\"y\")\n    x.add(\"s\", 1)\n    z = x + y\n    z.add(\"t\", 2)\n    assert x.duration() == 1 and z.duration() == 3, \"+ should make a new playlist without changing the old ones\"")

### Puzzle: A car with an engine

Arrange the lines so that a `Car` has an `Engine` and delegates `start` to it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Box:
    def __init__(self):
        self.items = []

class Wrapper:
    def __init__(self, inner):
        self.inner = inner

    def add(self, x):
        self.inner.items.append(x)

b = Box()
w1 = Wrapper(b)
w2 = Wrapper(b)
w1.add(1)
w2.add(2)

p = ___   # b.items
q = ___   # w1.inner is w2.inner
r = ___   # isinstance(w1, Box)
s = ___   # hasattr(w1, "items")
t = ___   # len(w2.inner.items)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2], \"p\"\n    assert (q, r, s, t) == (True, False, False, 2)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a thermostat from parts

A thermostat is built from three parts, each of which can be replaced:

- a **sensor** with `read()`, returning the temperature in °C;
- a **schedule** with `target(hour)`, returning the wanted temperature at that hour (0 to 23);
- a **heater** with `on()`, `off()` and an attribute `is_on`.

Write:

- `ListSensor(readings)`: `read()` returns the readings one after another; after the last one it keeps returning the last. An empty list raises `ValueError` when the sensor is made.
- `DaySchedule(day, night, start=7, end=22)`: the target is `day` from `start` up to but not including `end`, and `night` otherwise.
- `Heater()`: starts off; `on()` and `off()` set `is_on`, and `switches` counts how many times it actually **changed** (turning on a heater that's already on doesn't count).
- `Thermostat(sensor, schedule, heater)` with `tick(hour)`: read the temperature once; if it's below the target by **more than** 0.5, turn the heater on; if it's above by more than 0.5, turn it off; otherwise leave it. Return `"on"`, `"off"` or `"hold"` for what it decided. Record each tick in a list `history` as `(hour, reading, target, decision)`.

The thermostat must work with any objects that have the right methods, and changing `thermostat.schedule` must take effect at the next tick.

In [ ]:
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parts():\n    \"the parts\"\n    LS, DS, H = ns[\"ListSensor\"], ns[\"DaySchedule\"], ns[\"Heater\"]\n    s = LS([18.0, 19.5])\n    assert [s.read() for _ in range(4)] == [18.0, 19.5, 19.5, 19.5]\n    try:\n        LS([])\n        assert False, \"an empty list of readings should raise ValueError\"\n    except ValueError:\n        pass\n    d = DS(21, 16)\n    assert [d.target(h) for h in [0, 6, 7, 12, 21, 22, 23]] == [16, 16, 21, 21, 21, 16, 16]\n    assert DS(20, 15, start=9, end=17).target(8) == 15\n    h = H()\n    assert h.is_on is False and h.switches == 0\n    h.on(); h.on(); h.off(); h.off(); h.on()\n    assert h.is_on is True and h.switches == 3\ndef test_thermostat():\n    \"deciding\"\n    LS, DS, H, T = ns[\"ListSensor\"], ns[\"DaySchedule\"], ns[\"Heater\"], ns[\"Thermostat\"]\n    heater = H()\n    t = T(LS([19.0, 20.6, 21.6, 21.4, 15.6]), DS(21, 16), heater)\n    got = [t.tick(h) for h in [7, 8, 9, 10, 23]]\n    assert got == [\"on\", \"hold\", \"off\", \"hold\", \"hold\"], f\"got {got}\"\n    assert heater.switches == 2 and heater.is_on is False\n    assert t.history[0] == (7, 19.0, 21, \"on\") and t.history[4] == (23, 15.6, 16, \"hold\") and len(t.history) == 5\ndef test_swapping():\n    \"duck typing and swapping parts\"\n    H, T, DS = ns[\"Heater\"], ns[\"Thermostat\"], ns[\"DaySchedule\"]\n    class Fixed:\n        def __init__(self, c):\n            self.c = c\n        def read(self):\n            return self.c\n    class Always:\n        def __init__(self, c):\n            self.c = c\n        def target(self, hour):\n            return self.c\n    t = T(Fixed(18.0), Always(18.0), H())\n    assert t.tick(3) == \"hold\"\n    t.schedule = Always(25)\n    assert t.tick(3) == \"on\" and t.heater.is_on\n    t.sensor = Fixed(30)\n    assert t.tick(4) == \"off\"")

## Warm-up

Making `Queue` a child of `list` gave it `join` and `serve` in a few lines, but it also gave it **everything else** a list can do. Anyone can `insert` at the front and jump the queue, and `q + [...]` hands back a plain `list`, not a `Queue`. A queue isn't really a list with extras; it's something that **uses** a list to keep its order. Today's topic is building classes that way.

## Has a, not is a

**Composition** means building an object from other objects that it holds as attributes, and doing its work by calling theirs. The queue, rebuilt:

- The queue offers exactly the operations a queue should have, and nothing else. Nobody can jump in at the front.
- The list is an internal detail (single underscore, as in Week 29). It could become a `collections.deque` (Week 20), which is faster at removing from the front, and no code that uses `Queue` would notice.
- Passing a call on to a part, as `serve` does with `pop`, is called **delegation**.

The question that decides it is the one from Day 1. Say the two sentences out loud:

| Sentence | True? | So |
|---|---|---|
| A queue **is a** list | no: you can't insert or sort a queue | not inheritance |
| A queue **has a** list | yes | composition |
| An ebook **is a** book | yes, everywhere a book works | inheritance is fine |
| A car **is an** engine | no | not inheritance |
| A car **has an** engine | yes | composition |

Composition is everywhere in real programs. A shopping basket has items; a football league has teams, which have players; a train has carriages. Each part is its own class, tested on its own, and the bigger object puts them together:

## Swappable parts

A part held in an attribute can be **replaced**, even while the program runs. That gives you a kind of flexibility inheritance can't. An object's class is fixed when it's made, but its parts aren't.

Here's a design problem. A shop's orders can be priced normally, with a student discount, or with a bulk discount. With inheritance you'd write `StudentOrder`, `BulkOrder`, and so on. Then a student places a bulk order. And what about an order that becomes a student order after the customer shows their card?

With composition, the **pricing rule** is a part of the order:

- Each pricing rule is a small class with one method, `price`, and `Order` only cares that its part has that method (Day 4's duck typing).
- A new rule is one new class, and works with every order.
- Notice `pricing=None` and then `FullPrice()` inside: a default argument is made once, when the function is defined (Week 28, Day 3), so a shared default object would be shared by every order. For a pricing rule with no state it wouldn't matter, but it's a good habit.

This idea, an object holding a swappable part that does one job a particular way, is common enough to have a name: the **strategy** pattern. Week 35 meets it again, among other design patterns.

Composition also avoids a problem called **class explosion**. Suppose messages can go by email or text, and each can be logged or not, and urgent or not. With inheritance, every combination is a class: `Email`, `LoggedEmail`, `UrgentEmail`, `LoggedUrgentEmail`, and the same again for texts, eight in all, and every new option doubles it. With composition there are two channel classes and two option settings, combined when an object is made.

## When inheritance hurts

Inheritance ties a child closely to its parent's insides. Three problems follow when it's used where "is a" isn't true:

1. **The child gets everything**, wanted or not, like the queue's `insert`. You can't un-inherit a method; overriding it to raise an error breaks Day 3's rule that a child must work wherever its parent does.
2. **Changes in the parent can break the child.** A child often depends on details of how the parent does its work (which method calls which), and a parent written by someone else can change those details.
3. **Built-in types don't always use your overrides.** This one surprises everyone:

Only the key set with `d[...] = ...` was made lower case. The `dict` constructor, `update` and `get` are written in C for speed and go straight to the stored data without calling your methods. To make this work by inheritance you'd have to override every one of them. (The standard library offers `collections.UserDict` and `UserList`, which are written so that overrides are respected, for exactly this reason.) With composition, the class controls every way in:

Week 31 shows how to let your own classes use `d[key]`, `in` and `len()` through special methods, so a composed class can feel just like a built-in one.

## Wrapping an object

A useful kind of composition is the **wrapper**: an object that holds another object with the same methods, passes the calls on, and adds something along the way. The wrapper has the same interface as what it wraps, so it can be used anywhere the original could, and wrappers can be stacked:

The order of wrapping decides the result: in the last line, the signature is added first and then shouted. Compare Day 3's mixins, which did a similar job by inheritance. Mixins fix the combination when the class is written; wrappers choose it when the object is made.

## Choosing

Neither tool is always right. Ask these questions about the child or the part:

| Question | If yes |
|---|---|
| Is it true that every X **is a** Y, and X works everywhere Y does? | inheritance is reasonable |
| Does X just want to **reuse some code** from Y? | composition, or a helper function |
| Does X **have** a Y, or use one to do its work? | composition |
| Should the behaviour **change while the program runs**, or be mixed and matched? | composition |
| Is Y a built-in type such as `list` or `dict`? | usually composition |
| Is it a **family of exceptions**, or a framework that asks you to subclass? | inheritance |

A short version that serves well: **prefer composition, and inherit when "is a" is really true**. Shallow families (one or two levels) are easier to follow than deep ones.

## Practice

## Quiz

## Daily challenge

## Before you go

Look back at this week's examples and find one that used inheritance where composition would do, then sketch (in a few lines) how the composed version would look. Tomorrow is the Forge: a week of inheritance, `super()`, the MRO, duck typing and composition, put together.